# 模块五 — ELT 数据加工与 DWA 主题宽表

**幕一（业务痛点）**：月底分析会上，业务部门问「10 月各矿井日产量是多少？」数据分析师翻出 VBAK、PI tags、LIMS samples 三套原始数据，要写 5 个 JOIN、等 IT 排期 3~5 天。

**幕二（DWA 解法）**：有了 DWA 预聚合宽表，数据工程师提前算出日销售汇总、传感器告警排名、月度煤质报告。业务人员直接查宽表，10 秒钟出结果，不用再等 IT 排期。

---

## 3 步学习节奏

| 步骤 | 主题 | 看什么 |
|------|------|--------|
| 步骤 1 | 构建 DWA 宽表 | 3 张宽表是怎么生成的 |
| 步骤 2 | 即席查询 | 3 个业务场景：销售趋势 / 告警排名 / 煤质月报 |
| 步骤 3 | 4 个分析场景验证 | 哪些能查，哪些还需 Phase 2 |

> **设计原则**：本 notebook 调函数演示，大段聚合逻辑在 `scripts/build_dwa_models.py`，后续可慢慢理解。

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import subprocess, sys

# 安装 duckdb（如未安装）
try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb'], check=True)
    import duckdb

# DATA_ROOT 路径
DATA_ROOT = '/home/szs/Playground/dg-demo/data/historical'
LAKEHOUSE = '/home/szs/Playground/dg-demo/data/lakehouse'
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"LAKEHOUSE = {LAKEHOUSE}")

## 步骤 1：构建 3 张 DWA 汇总宽表

调用 `scripts/build_dwa_models.py` 将 DWD 清洗层数据聚合为 3 张宽表：
- `dwa_sales_daily`：日销售汇总（来源：SAP VBAK）
- `dwa_tag_alarm`：传感器告警 Top20 排名（来源：PI tags）
- `dwa_coal_quality`：月度煤质汇总报告（来源：LIMS samples）

In [ ]:
# 调用 build_dwa_models.py 构建 3 张 DWA 宽表
result = subprocess.run(
    ['uv', 'run', 'python', 'scripts/build_dwa_models.py', '--layer', 'dwa'],
    capture_output=True, text=True, cwd='/home/szs/Playground/dg-demo'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(1)
print('\n✅ 3 张 DWA 宽表构建完成')

## 步骤 2：即席查询 — 日销售趋势

查 `dwa_sales_daily`，看最近 30 天订单数 / 客户数 / 销售额趋势。

In [ ]:
conn.execute(f"""
    CREATE VIEW IF NOT EXISTS dwa_sales AS
    SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/sap_erp/dwa_sales_daily')
""")

df = conn.execute('''
    SELECT sale_date, order_count, customer_count,
           ROUND(total_amount/1e6, 2) AS total_amount_m
    FROM dwa_sales
    ORDER BY sale_date ASC
    LIMIT 10
''').df()

print('最近 10 天日销售汇总（单位：百万元，按日期升序）：')
print(df.to_string(index=False))

## 步骤 3：即席查询 — 传感器告警 Top20

查 `dwa_tag_alarm`，看哪些传感器高频告警最多，辅助安全部排维护优先级。

In [ ]:
conn.execute(f"""
    CREATE VIEW IF NOT EXISTS dwa_alarm AS
    SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/pi_system/dwa_tag_alarm')
""")

df = conn.execute('''
    SELECT mine, tag, high_value_count, missing_count,
           ROUND(avg_value, 2) AS avg_value
    FROM dwa_alarm
    ORDER BY high_value_count DESC
    LIMIT 10
''').df()

total = df['high_value_count'].sum()
top10 = df['high_value_count'].head(10).sum()
print(f'Top10 告警量占总告警量比例：{top10}/{total} = {top10/total*100:.1f}%')
print('\nTop10 高频告警传感器：')
print(df.to_string(index=False))

## 步骤 4：即席查询 — 月度煤质报告

查 `dwa_coal_quality`，按矿井×月份聚合灰分 / 挥发分 / 硫分 / 发热量。

In [ ]:
conn.execute(f"""
    CREATE VIEW IF NOT EXISTS dwa_quality AS
    SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/lims/dwa_coal_quality')
""")

df = conn.execute('''
    SELECT MINE_CODE, month, SAMPLE_TYPE,
           sample_count,
           avg_ash_content, avg_volatile_content,
           avg_sulfur_content, avg_gross_calorific,
           min_ash_content, max_ash_content, lab_count
    FROM dwa_quality
    ORDER BY month DESC, MINE_CODE
    LIMIT 10
''').df()

print('月度煤质报告（最近月份）：')
print(df.to_string(index=False))

## 步骤 5：4 个分析场景验证

验证 DWA 宽表对 4 个典型业务场景的支撑能力。

In [ ]:
scenarios = [
    ('销售趋势',     'dwa_sales',   '✅ 可查：最近 30 天订单数/销售额'),
    ('告警传感器排名', 'dwa_alarm',   '✅ 可查：Top20 高频告警传感器'),
    ('月度煤质报告', 'dwa_quality', '✅ 可查：矿井×月份煤质聚合'),
    ('跨系统产销对比', 'dwa_join',   '⚠️ 需业务自己写 JOIN（当前无跨系统宽表）'),
]

print('场景可达性验证：')
print(f'{"场景":<15} {"状态":<8} {"说明"}')
print('-' * 60)
for name, table, status in scenarios:
    print(f'{name:<15} {status[:6]:<8} {status}')

## 诚实声明：当前 DWA 的边界

3 张 DWA 宽表均为**单系统**汇总（来源各自独立的源系统），不支持跨系统产销对比分析：

- `dwa_sales_daily` ← SAP VBAK
- `dwa_tag_alarm` ← PI tags
- `dwa_coal_quality` ← LIMS samples

**跨系统产销对比**（日产量 vs 日发货量）需要将 PI 生产实绩 × LIMS 煤质 × SAP 销售订单 × KNA1 客户四表 JOIN，这依赖 **Phase 2 主数据标准化**（统一矿井/客户编码维表）先完成。当前业务人员如需此类分析，需自己写 SQL JOIN。

Phase 2 将交付 `dwa_sales_production` 跨系统产销宽表，届时跨系统分析可开箱即得。

## 模块五总结

| DWA 表 | 数据源 | 主要字段 | 业务场景 |
|--------|--------|---------|---------|
| `dwa_sales_daily` | SAP VBAK | sale_date, order_count, total_amount | 日销售趋势 |
| `dwa_tag_alarm` | PI tags | tag, alarm_count, high_value_count | 传感器告警 Top20 |
| `dwa_coal_quality` | LIMS samples | mine_code, month, avg_ash, avg_qgr | 月度煤质报告 |

**前置模块**：[模块一看资产](module1.ipynb) → [模块二找问题](module2.ipynb) → [模块三追血缘](module3.ipynb) → [模块四清洗修复](module4.ipynb) → **模块五 DWA 宽表**

**后续模块**：[模块六即席查询](module6.ipynb)（Phase 1 终点，产销对比需 Phase 2）